# Reliability — how stable is the comparison across runs?

The comparison is stochastic: the compass and the two per-log models start from random weights
and are fitted by a random-order full-batch descent. Two runs on the *same* two logs therefore do
not return the same numbers. This notebook asks how far apart they land.

Per comparison pair `L1` vs `L2`, the whole embedding comparison is repeated `RUNS` times, each
run seeded differently, and every run yields one similarity `sim(a)` per shared activity. The
runs are then compared to each other:

1. **Similarity distance** — for every unordered pair of runs, the mean absolute difference of
   their similarity vectors, `mean_a |sim_r(a) - sim_s(a)|`, averaged over all run pairs. Reads
   in the units of the similarity itself: 0.02 means two runs disagree by two hundredths of a
   cosine on the average activity.
2. **Rank agreement** — Spearman's rho between the two runs' similarity vectors, averaged over
   all run pairs. This is what actually matters downstream: not whether the similarities are
   numerically identical, but whether the runs put the activities in the same order, since it is
   the ordering that decides which activities `01_validity` selects.

No projection and no EMSC here — this is purely about the embedding stage. A pair is reduced to
**one summary row**, written to disk, and everything it allocated is dropped before the next pair
starts.

## Configuration

`RECOMPUTE` decides whether the pair loop runs at all. With `False` nothing is trained: the
results section simply reads the per-pair rows already on disk.

`RUNS` is how often each pair is repeated. Run `r` is seeded `SEED + r`, so the whole table is
reproducible, and the same seeds are used for every pair.

Settings shared by the whole experiment (`CONTEXT_WINDOW`, `EMBEDDING_DIM`, `UNIT_NORM`,
`BALANCE_COMPASS`) are globals and match `01_validity`, so the numbers here describe that
experiment's embedding stage. Both metrics are taken over all shared activities, so no quantile
enters this notebook. `PAIRS` and its per-pair training schedules are the
same list as well — a pair is only as reliable as the schedule it was tuned with.

In [34]:
# --- Imports -----------------------------------------------------------------
import gc
import os
from itertools import combinations

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

import pm4py
import log_comparison_cwindow as lcw

# --- Fixed settings ----------------------------------------------------------
SEED = 42  # run r of every pair is seeded SEED + r
DEVICE = "cpu"  # small full-batch runs; MPS is slower for these
LOGS_DIR = "logs"  # a log named X is read from LOGS_DIR/X.xes
RESULTS_DIR = "results/02_reliability"  # one CSV per pair, one row each

# False = do not train anything; just read the rows already in RESULTS_DIR
RECOMPUTE = True

RUNS = 10  # repetitions of the comparison per pair
if RUNS < 2:
    raise ValueError(
        f"RUNS must be >= 2 to compare runs against each other, got {RUNS}."
    )

# --- Global experiment settings (identical for every pair, and to 01_validity) ---
CONTEXT_WINDOW = 4  # context window c
EMBEDDING_DIM = 32  # d, the activity embedding dimension
BALANCE_COMPASS = (
    True  # scale the smaller log's case counts by the size imbalance so both
)
#                         logs weigh equally in the compass loss; nothing is resampled, and
#                         the per-log retraining stays unweighted
UNIT_NORM = True  # keep the activity embeddings on the unit sphere: each context
#                         embedding is L2-normalised before it enters the concatenated context
#                         vector, and the read-out is normalised too. The model cannot spend
#                         capacity on embedding length, so an activity is described by the
#                         direction alone -- which is exactly what the cosine reads

# --- One comparison entry ----------------------------------------------------
# Every pair states its own full set of per-pair hyperparameters, so any pair can be tuned
# without touching the others. Nothing is inherited: a missing or misspelled hyperparameter
# raises instead of silently falling back to a default.
#
#   compass_learning_rate / compass_epochs    compass stage (the shared space)
#   log_learning_rate / log_epochs            per-log frozen-output retraining
HYPERPARAM_KEYS = (
    "compass_learning_rate",
    "compass_epochs",
    "log_learning_rate",
    "log_epochs",
)


def pair(l1, l2, **hyperparams):
    """One comparison: the two log names plus every hyperparameter it runs with."""
    missing = [k for k in HYPERPARAM_KEYS if k not in hyperparams]
    unknown = [k for k in hyperparams if k not in HYPERPARAM_KEYS]
    if missing or unknown:
        raise ValueError(
            f"{l1} vs {l2}: "
            + (f"missing hyperparameters {missing}. " if missing else "")
            + (f"unknown hyperparameters {unknown}." if unknown else "")
        )
    return {"l1": l1, "l2": l2, **hyperparams}


def pair_id(p):
    return f"{p['l1']} vs {p['l2']}"


def pair_path(p):
    """The file this pair's single result row is written to."""
    return f"{RESULTS_DIR}/{p['l1']}__{p['l2']}.csv"


# --- The experiment ----------------------------------------------------------
PAIRS = [
    # --- BPIC11 age ---
    pair(
        "BPIC11_age_low",
        "BPIC11_age_high",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    # --- BPIC15 municipalities ---
    pair(
        "BPIC15_M1",
        "BPIC15_M2",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M1",
        "BPIC15_M3",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M1",
        "BPIC15_M4",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M1",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M2",
        "BPIC15_M3",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M2",
        "BPIC15_M4",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M2",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M3",
        "BPIC15_M4",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M3",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M4",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    # --- BPIC18 departments ---
    pair(
        "BPIC18_D4e",
        "BPIC18_D6b",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D4e",
        "BPIC18_Dd4",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D4e",
        "BPIC18_De7",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D6b",
        "BPIC18_Dd4",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D6b",
        "BPIC18_De7",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_Dd4",
        "BPIC18_De7",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
]

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"{len(PAIRS)} pairs x {RUNS} runs: {', '.join(pair_id(p) for p in PAIRS)}")

17 pairs x 10 runs: BPIC11_age_low vs BPIC11_age_high, BPIC15_M1 vs BPIC15_M2, BPIC15_M1 vs BPIC15_M3, BPIC15_M1 vs BPIC15_M4, BPIC15_M1 vs BPIC15_M5, BPIC15_M2 vs BPIC15_M3, BPIC15_M2 vs BPIC15_M4, BPIC15_M2 vs BPIC15_M5, BPIC15_M3 vs BPIC15_M4, BPIC15_M3 vs BPIC15_M5, BPIC15_M4 vs BPIC15_M5, BPIC18_D4e vs BPIC18_D6b, BPIC18_D4e vs BPIC18_Dd4, BPIC18_D4e vs BPIC18_De7, BPIC18_D6b vs BPIC18_Dd4, BPIC18_D6b vs BPIC18_De7, BPIC18_Dd4 vs BPIC18_De7


## Helpers

`load_log` reads one log; the two metric helpers reduce a set of runs to the numbers the summary
reports. Nothing is cached — a log is read again for every pair it takes part in, so no pair's
data outlives its iteration.

In [35]:
CASE_COL, ACTIVITY_COL, TIME_COL = "case:concept:name", "concept:name", "time:timestamp"


def load_log(name):
    """Read the log named ``name`` from ``LOGS_DIR``, reduced to the three needed columns.

    ``concept:name`` is the activity as ``00_preprocessing`` left it -- for BPIC15 that is the
    readable ``activityNameEN`` label, not the internal activity code, which splits one activity
    into several numbered variants. Nothing is relabelled here.
    """
    return pm4py.read_xes(f"{LOGS_DIR}/{name}.xes")[[CASE_COL, ACTIVITY_COL, TIME_COL]]


def pairwise_distances(sims):
    """Mean absolute difference of the similarity vectors, for every pair of runs.

    :param sims: ``[n_activities, RUNS]``; column r is run r's similarity per activity.
    :returns: one distance per unordered run pair, in run-pair order.
    """
    return np.array(
        [
            np.abs(sims[:, r] - sims[:, s]).mean()
            for r, s in combinations(range(sims.shape[1]), 2)
        ]
    )


def pairwise_spearman(sims):
    """Spearman rank correlation of the similarity vectors, for every pair of runs.

    Rank rather than value agreement: what the downstream selection depends on is the order the
    activities come out in, not the cosines themselves.

    :param sims: ``[n_activities, RUNS]``; column r is run r's similarity per activity.
    :returns: one rho per unordered run pair, in run-pair order.
    """
    rho = sims.corr(method="spearman").to_numpy()
    return rho[np.triu_indices(rho.shape[0], k=1)]

## The run — one pair at a time

For each pair: load the two logs once, then repeat the embedding comparison `RUNS` times on them,
keeping only each run's similarity Series and its two selected sets. The runs are then reduced to
the summary numbers, the row is written to `RESULTS_DIR`, and the logs are freed.

The shared-activity index is fixed by the two logs, not by the fitting, so every run returns the
same activities in the same order and the run vectors line up directly.

In [36]:
def run_pair(p):
    """Run one comparison RUNS times and return the single row describing their agreement."""
    l1, l2 = p["l1"], p["l2"]
    log_1, log_2 = load_log(l1), load_log(l2)

    run_sims = []
    for run in tqdm(range(RUNS), desc=pair_id(p), leave=False):
        lcw.set_seed(SEED + run)  # the only difference between the runs
        model_1, model_2, _ = lcw.compare_cwindow_event_logs(
            log_1,
            log_2,
            names=(l1, l2),
            c=CONTEXT_WINDOW,
            embedding_dim=EMBEDDING_DIM,
            compass_learning_rate=p["compass_learning_rate"],
            compass_epochs=p["compass_epochs"],
            log_learning_rate=p["log_learning_rate"],
            log_epochs=p["log_epochs"],
            balance_compass=BALANCE_COMPASS,
            unit_norm=UNIT_NORM,
            device=DEVICE,
            verbose=False,
        )

        run_sims.append(lcw.activity_similarity(model_1, model_2))
        del model_1, model_2

    # one column per run; the index is fixed by the two logs, so the runs line up directly
    sims = pd.concat(run_sims, axis=1, keys=range(RUNS))  # [n_activities, RUNS]
    distances = pairwise_distances(sims.to_numpy())
    rhos = pairwise_spearman(sims)

    acts_1, acts_2 = (
        set(log_1[ACTIVITY_COL].unique()),
        set(log_2[ACTIVITY_COL].unique()),
    )
    row = {
        "L1": l1,
        "L2": l2,
        "runs": RUNS,
        "shared acts": len(acts_1 & acts_2),
        "total acts": len(acts_1 | acts_2),
        "sim dist mean": float(distances.mean()),
        "sim dist max": float(distances.max()),
        "spearman mean": float(rhos.mean()),
        "spearman min": float(rhos.min()),
    }

    del log_1, log_2, run_sims, sims
    return row


if RECOMPUTE:
    for p in tqdm(PAIRS, desc="Comparisons"):
        row = run_pair(p)
        pd.DataFrame([row]).to_csv(
            pair_path(p), index=False
        )  # saved before the next pair
        print(
            f"{pair_id(p)}: sim distance {row['sim dist mean']:.4f}, spearman "
            f"{row['spearman mean']:.3f} (worst {row['spearman min']:.3f}) "
            f"-> {pair_path(p)}"
        )
        del row
        gc.collect()
else:
    print(f"RECOMPUTE = False — reading the rows already in {RESULTS_DIR}/")

Comparisons:   0%|          | 0/17 [00:00<?, ?it/s]

BPIC11_age_low vs BPIC11_age_high:   0%|          | 0/10 [00:00<?, ?it/s]

BPIC11_age_low vs BPIC11_age_high: sim distance 0.0520, spearman 0.957 (worst 0.950) -> results/02_reliability/BPIC11_age_low__BPIC11_age_high.csv


BPIC15_M1 vs BPIC15_M2:   0%|          | 0/10 [00:00<?, ?it/s]

BPIC15_M1 vs BPIC15_M2: sim distance 0.0491, spearman 0.950 (worst 0.943) -> results/02_reliability/BPIC15_M1__BPIC15_M2.csv


...

BPIC18_D6b vs BPIC18_Dd4: sim distance 0.0267, spearman 0.964 (worst 0.937) -> results/02_reliability/BPIC18_D6b__BPIC18_Dd4.csv


BPIC18_D6b vs BPIC18_De7:   0%|          | 0/10 [00:00<?, ?it/s]

BPIC18_D6b vs BPIC18_De7: sim distance 0.0260, spearman 0.956 (worst 0.927) -> results/02_reliability/BPIC18_D6b__BPIC18_De7.csv


BPIC18_Dd4 vs BPIC18_De7:   0%|          | 0/10 [00:00<?, ?it/s]

BPIC18_Dd4 vs BPIC18_De7: sim distance 0.0291, spearman 0.945 (worst 0.909) -> results/02_reliability/BPIC18_Dd4__BPIC18_De7.csv


## Results

The single-row files are read back and stacked, one row per comparison pair.

* **runs** — repetitions behind the row; the numbers average over its `runs * (runs - 1) / 2`
  run pairs.
* **shared / total acts** — activity alphabet overlap of the two logs.
* **sim dist mean / max** — average and worst-case distance between two runs' similarity vectors,
  in cosine units. Lower is more reliable.
* **spearman mean / min** — average and worst-case Spearman's rho between two runs' similarity
  vectors, over all shared activities. 1.0 = every run ranks the activities identically, 0 = the
  runs carry no shared ordering at all.

In [37]:
paths = [pair_path(p) for p in PAIRS if os.path.exists(pair_path(p))]
missing = [pair_id(p) for p in PAIRS if not os.path.exists(pair_path(p))]
if missing:
    print(f"no result row for {missing} — set RECOMPUTE = True and rerun")
if not paths:
    raise RuntimeError(f"{RESULTS_DIR}/ holds no result rows")

results = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
results.to_csv("results/02_reliability_summary.csv", index=False)

display(
    results.style.format(
        {
            "sim dist mean": "{:.4f}",
            "sim dist max": "{:.4f}",
            "spearman mean": "{:.3f}",
            "spearman min": "{:.3f}",
        }
    )
    .background_gradient(
        subset=["spearman mean", "spearman min"], cmap="RdYlGn", vmin=0, vmax=1
    )
    .background_gradient(subset=["sim dist mean"], cmap="RdYlGn_r")
    .hide(axis="index")
)

L1,L2,runs,shared acts,total acts,sim dist mean,sim dist max,spearman mean,spearman min
BPIC11_age_low,BPIC11_age_high,10,367,624,0.0520,0.0571,0.957,0.950
BPIC15_M1,BPIC15_M2,10,262,331,0.0491,0.0543,0.950,0.943
BPIC15_M1,BPIC15_M3,10,250,316,0.0442,0.0506,0.950,0.938
BPIC15_M1,BPIC15_M4,10,256,305,0.0481,0.0532,0.939,0.924
BPIC15_M1,BPIC15_M5,10,253,321,0.0470,0.0545,0.945,0.930
BPIC15_M2,BPIC15_M3,10,256,325,0.0479,0.0552,0.950,0.940
BPIC15_M2,BPIC15_M4,10,258,318,0.0499,0.0564,0.945,0.935
BPIC15_M2,BPIC15_M5,10,252,337,0.0463,0.0518,0.955,0.942
BPIC15_M3,BPIC15_M4,10,257,292,0.0502,0.0558,0.947,0.937
BPIC15_M3,BPIC15_M5,10,254,308,0.0507,0.0566,0.942,0.930
